[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day4_lecture.ipynb)

# Day 4 · 강의 — 딥러닝

클래스 · 텐서 · 학습 루프 · 평가

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

수업을 따라가며 진행한다.

**실습** 셀은 그대로 실행해 결과를 눈으로 확인한다.
**문제** 셀은 수업 중에 같이 푼다.

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 클래스

**실습.** 클래스는 설계도, 인스턴스는 그 설계도로 만든 물건이다.

In [ ]:
class Batch:
    def __init__(self, bid, temp):
        self.bid = bid
        self.temp = temp

    def is_hot(self):
        return self.temp >= 880

b = Batch("B00115", 898.9)
print(b.bid, b.temp, b.is_hot())

**실습.** 상속은 부모의 기능을 물려받고 필요한 것만 고쳐 쓴다.

In [ ]:
class Machine:
    def __init__(self, name):
        self.name = name
    def describe(self):
        return f"{self.name} 설비"

class Kiln(Machine):
    def __init__(self, name, rated):
        super().__init__(name)        # 부모 초기화
        self.rated = rated
    def describe(self):               # 재정의
        return f"{self.name} 소성로 (정격 {self.rated}°C)"

print(Machine("A").describe())
print(Kiln("C", 870).describe())

> **빈칸 문제 1.** `Batch` 에 **최적 온도(890)에서 벗어난 정도**를 돌려주는 `gap()` 메서드를 넣는다.

In [ ]:
class Batch:
    def __init__(self, bid, temp):
        self.bid = bid
        self.temp = temp

    def gap(self):
        return ___

b = Batch('B00115', 898.9)

assert abs(b.gap() - 8.9) < 1e-6, f'실제 {b.gap()}'
print('통과')

## 2. 텐서

**실습.** 텐서는 NumPy 배열과 거의 같다. GPU 로 옮길 수 있고 기울기를 기억한다.

In [ ]:
import torch

x = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
print(x.shape, x.dtype)
print(x * 2)
print(x @ torch.tensor([[1.], [1.], [1.]]))

print(torch.zeros(2, 3).shape)
print(x.numpy().shape)      # NumPy 로 되돌리기

**실습.** requires_grad 를 켜면 계산 과정을 기억했다가 기울기를 돌려준다.

In [ ]:
import torch

w = torch.tensor(3.0, requires_grad=True)
loss = (w - 5) ** 2      # 최솟값은 w = 5
loss.backward()
print(w.grad)            # d(loss)/dw = 2(w-5) = -4

> **빈칸 문제 2.** `a` 를 3행 2열 텐서로 바꿔 `b` 에 담는다.

In [ ]:
import torch
a = torch.arange(6, dtype=torch.float32)
b = ___

assert b.shape == (3, 2), f'기대 (3, 2), 실제 {tuple(b.shape)}'
print('통과')

## 3. 모델과 학습 루프

**실습.** nn.Module 을 상속해 층을 쌓고, forward 에 흐르는 순서를 적는다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))

class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 16)
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc2(h)

X_tr, X_te, y_tr, y_te = tensors()
model = MLP(X_tr.shape[1])
print(model)
print(model(X_tr[:4]).shape)    # 학습 전에 shape 부터 확인한다

**실습.** 학습 루프는 다섯 줄이다. 순서가 정해져 있다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))

class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

X_tr, X_te, y_tr, y_te = tensors()
model = MLP(X_tr.shape[1])
lossfn = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(1, 101):
    pred = model(X_tr)             # 1. 예측
    loss = lossfn(pred, y_tr)      # 2. 손실
    opt.zero_grad()                # 3. 기울기 비우기
    loss.backward()                # 4. 역전파
    opt.step()                     # 5. 갱신
    if epoch % 25 == 0:
        print(f"{epoch:>4}  loss {loss.item():.4f}")

> **빈칸 문제 3.** 은닉층을 **32개**로 키운 `MLP` 를 만들고, 입력 4건을 통과시킨 출력 모양을 확인한다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
X_tr, X_te, y_tr, y_te = tensors()
class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.fc1 = ___
        self.fc2 = ___
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

model = MLP(X_tr.shape[1])
out = model(X_tr[:4])

assert tuple(out.shape) == (4, 1), f'기대 (4, 1), 실제 {tuple(out.shape)}'
assert model.fc1.out_features == 32, '은닉층이 32여야 한다'
print('통과')

> **빈칸 문제 4.** 학습 루프 다섯 줄의 **순서를 채운다.** 100회 돌린 뒤 손실이 줄었는지 본다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

X_tr, X_te, y_tr, y_te = tensors()
model = MLP(X_tr.shape[1])
lossfn = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
first = None
for epoch in range(100):
    pred = model(X_tr)
    loss = lossfn(pred, y_tr)
    ___
    ___
    ___
    if first is None: first = loss.item()
last = loss.item()

assert last < first, f'손실이 줄어야 한다: {first:.4f} → {last:.4f}'
print(f'통과 — {first:.4f} → {last:.4f}')

## 4. 평가

**실습.** 출력은 로짓이다. 확률로 바꾸려면 시그모이드를 통과시킨다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))

class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

X_tr, X_te, y_tr, y_te = tensors()
model = MLP(X_tr.shape[1])
lossfn = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
for _ in range(300):
    loss = lossfn(model(X_tr), y_tr)
    opt.zero_grad(); loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    prob = torch.sigmoid(model(X_te))
    pred = (prob >= 0.5).float()
    acc = (pred == y_te).float().mean().item()
print('테스트 정확도', round(acc, 3))

## 5. 종합 문제

---

### 미니 프로젝트

여기까지가 나흘의 마지막이다. 남은 시간에는 **스켈레톤의 `# TODO` 여덟 곳**을 채운다.
제출물은 노트북 링크 하나이고, 기준은 점수가 아니라 **런타임 초기화 후 끝까지 도는지**다.